# TryFit Next — gerçek mankenli ürün testi
Bu notebook kendi Colab GPU oturumunuzda FASHN VTON v1.5 çalıştırır. Ücretli API veya HF Space çağrısı yapmaz.

**Araştırma/değerlendirme kullanımı:** VTON Apache 2.0; resmî human parser NVIDIA SegFormer araştırma/değerlendirme lisansını devralır. Ticari dağıtım onayı değildir.

1. Colab: Runtime → Change runtime type → ücretsiz GPU. GPU tahsisi garanti değildir.
2. Kurulum hücrelerini çalıştırın. Yeniden başlatma istenirse oturumu yeniden başlatıp ilk hücreden devam edin.
3. Yerel manifesti gerçek ürün URL’leriyle doldurun; izinli `person.jpg` ve on ürün fotoğrafını hazırlayın.
4. Yükleme hücresinde fotoğrafları ve doldurduğunuz `manifest.json` dosyasını seçin.
5. Önce tek ürün, sonra on ürün deneyini çalıştırın. Sonuçları hemen indirin.

T4 BF16 desteklemez; resmî pipeline FP32 seçebilir. 8 GB iddiası T4 garantisi değildir. Bellek hatasında ücretsiz kaynakta çalışmadı olarak kaydedin.


In [ ]:
from pathlib import Path
import sys, subprocess, json, os
ROOT = Path('/content/tryfit-next')
ROOT.mkdir(parents=True, exist_ok=True)
def checked(*args):
    subprocess.run(list(args), check=True)
checked('nvidia-smi')
import torch
print('Python:', sys.version, 'Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('GPU tahsis edilmedi. Runtime ayarlarını kontrol edin.')
print('GPU:', torch.cuda.get_device_name(0), 'BF16:', torch.cuda.is_bf16_supported())
print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)


In [ ]:
UPSTREAM_COMMIT = '7c0f10af3f91ad4048fe9729c470a13ef905d25a'
vendor = ROOT / 'vendor/fashn-vton-1.5'
vendor.parent.mkdir(exist_ok=True)
if not vendor.exists():
    checked('git', 'clone', 'https://github.com/fashn-AI/fashn-vton-1.5.git', str(vendor))
checked('git', '-C', str(vendor), 'checkout', '--detach', UPSTREAM_COMMIT)
checked(sys.executable, '-m', 'pip', 'install', '-e', str(vendor))
checked(sys.executable, '-m', 'pip', 'check')


In [ ]:
# Runner bu notebook'a gömülüdür; başka bir TryFit deposu indirilmez.
(ROOT / 'experiment.py').write_text('"""Local, sequential research experiments. No remote inference or fallback images."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport html\nimport json\nimport platform\nimport re\nimport subprocess\nimport time\nimport traceback\nimport uuid\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom urllib.parse import urlparse\n\nMETRICS = ("garment_identity", "color_pattern", "cut", "face", "body", "arms_hands", "fabric", "source_pose")\nCATEGORIES = {"tops", "bottoms", "one-pieces"}\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef local_path(root, name):\n    path = (Path(root) / name).resolve()\n    if not path.is_relative_to(Path(root).resolve()):\n        raise ValueError("Girdi yolu proje klasörünün dışında olamaz.")\n    return path\n\n\ndef validate_manifest(manifest, root, selected_ids=None):\n    if manifest.get("schema_version") != 1:\n        raise ValueError("Desteklenmeyen manifest sürümü.")\n    person = manifest.get("person", {})\n    if person.get("consent_confirmed") is not True:\n        raise ValueError("Hedef fotoğraf için kullanım izni doğrulanmalı.")\n    if not local_path(root, person["image"]).is_file():\n        raise ValueError("Hedef kişi fotoğrafı bulunamadı.")\n    cases = manifest.get("cases", [])\n    ids = [case.get("id", "") for case in cases]\n    if len(ids) != len(set(ids)) or any(not re.fullmatch(r"[a-z0-9-]+", i) for i in ids):\n        raise ValueError("Deney kimlikleri benzersiz ve küçük harf/rakam/tire içermeli.")\n    if selected_ids and not set(selected_ids).issubset(ids):\n        raise ValueError("Seçilen deney kimliği manifestte yok.")\n    selected = [c for c in cases if not selected_ids or c["id"] in selected_ids]\n    if not selected:\n        raise ValueError("En az bir deney gerekli.")\n    for case in selected:\n        url = urlparse(case.get("product_url", ""))\n        host = (url.hostname or "").lower()\n        if url.scheme != "https" or not (host == "trendyol.com" or host.endswith(".trendyol.com")) or "-p-" not in url.path:\n            raise ValueError(f\'{case["id"]}: gerçek Trendyol ürün URL’si gerekli.\')\n        if case.get("category") not in CATEGORIES or case.get("garment_photo_type") not in {"model", "flat-lay"}:\n            raise ValueError(f\'{case["id"]}: kategori veya mankenli görsel türü geçersiz.\')\n        if not case.get("source_image_url") or not case.get("source_notes"):\n            raise ValueError(f\'{case["id"]}: kaynak görsel URL’si ve poz notu gerekli.\')\n        if not local_path(root, case["image"]).is_file():\n            raise ValueError(f\'{case["id"]}: ürün görseli bulunamadı.\')\n    return selected\n\n\ndef command_output(args):\n    result = subprocess.run(args, capture_output=True, text=True, check=False)\n    return {"exit_code": result.returncode, "stdout": result.stdout, "stderr": result.stderr}\n\n\ndef preflight():\n    import torch\n    import onnxruntime as ort\n    info = {\n        "python": platform.python_version(), "platform": platform.platform(),\n        "torch": torch.__version__, "torch_cuda": torch.version.cuda,\n        "cuda_available": torch.cuda.is_available(), "onnxruntime": ort.__version__,\n        "onnx_providers": ort.get_available_providers(),\n    }\n    if info["cuda_available"]:\n        info.update(gpu=torch.cuda.get_device_name(0),\n                    vram_gib=round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),\n                    bf16_supported=torch.cuda.is_bf16_supported())\n        info["expected_dtype"] = "bfloat16" if info["bf16_supported"] else "float32"\n    return info\n\n\ndef load_rgb(path):\n    from PIL import Image, ImageOps\n    with Image.open(path) as original:\n        return ImageOps.exif_transpose(original).convert("RGB")\n\n\ndef make_comparison(person, garment, output, destination):\n    from PIL import Image, ImageDraw, ImageOps\n    board = Image.new("RGB", (960, 512), "#f3f4f6")\n    draw = ImageDraw.Draw(board)\n    for index, (title, picture) in enumerate((("SOURCE GARMENT", garment), ("TARGET PERSON", person), ("GENERATED", output))):\n        thumb = ImageOps.contain(picture, (312, 476))\n        board.paste(thumb, (index * 320 + (320 - thumb.width) // 2, 28 + (476 - thumb.height) // 2))\n        draw.text((index * 320 + 8, 8), title, fill="black")\n    board.save(destination)\n\n\ndef run_experiment(root, manifest_path, weights_dir, selected_ids=None, *, seed=42, steps=30,\n                   guidance=1.5, segmentation_free=True, pipeline_factory=None):\n    """Raises on invalid inputs/setup; preserves individual inference failures in run.json."""\n    root = Path(root).resolve()\n    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8-sig"))\n    cases = validate_manifest(manifest, root, selected_ids)\n    # Decode every selected image before reserving a GPU pipeline.\n    person = load_rgb(local_path(root, manifest["person"]["image"]))\n    for case in cases:\n        load_rgb(local_path(root, case["image"]))\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")\n    run_dir = root / "runs" / f"{stamp}-{uuid.uuid4().hex[:8]}"\n    run_dir.mkdir(parents=True)\n    record = {"schema_version": 1, "created_at": stamp, "status": "initializing", "cases": [],\n              "quality_status": "not_reviewed", "settings": {\n                  "seed": seed, "num_timesteps": steps, "guidance_scale": guidance,\n                  "segmentation_free": segmentation_free, "num_samples": 1},\n              "manifest_sha256": sha256(manifest_path),\n              "person_sha256": sha256(local_path(root, manifest["person"]["image"]))}\n    write_json(run_dir / "manifest.json", manifest)\n    write_json(run_dir / "run.json", record)\n    try:\n        record["environment"] = preflight()\n        (run_dir / "pip-freeze.txt").write_text(command_output([__import__("sys").executable, "-m", "pip", "freeze"])["stdout"], encoding="utf-8")\n        record["upstream_commit"] = command_output(["git", "-C", str(root / "vendor/fashn-vton-1.5"), "rev-parse", "HEAD"])\n        if not record["environment"]["cuda_available"]:\n            raise RuntimeError("CUDA GPU yok; CPU üretimi başlatılmadı.")\n        if "CUDAExecutionProvider" not in record["environment"]["onnx_providers"]:\n            raise RuntimeError("ONNX CUDA provider yok; CUDA/cuDNN kurulumunu kontrol edin.")\n        weight_files = sorted(Path(weights_dir).rglob("*.safetensors")) + sorted(Path(weights_dir).rglob("*.onnx"))\n        record["weight_sha256"] = {str(p.relative_to(weights_dir)): sha256(p) for p in weight_files}\n        provenance = root / "model-provenance.json"\n        if provenance.exists():\n            record["model_provenance"] = json.loads(provenance.read_text(encoding="utf-8"))\n        write_json(run_dir / "run.json", record)\n        import torch\n        if pipeline_factory is None:\n            from fashn_vton import TryOnPipeline\n            pipeline = TryOnPipeline(weights_dir=str(weights_dir), device="cuda")\n        else:\n            pipeline = pipeline_factory(weights_dir)\n            record["pipeline_factory"] = pipeline_factory.__module__ + "." + pipeline_factory.__name__\n        record["actual_dtype"] = str(pipeline.inference_dtype)\n        record["status"] = "running"\n        for case in cases:\n            item = {"id": case["id"], "status": "running", "quality_status": "not_reviewed",\n                    "garment_sha256": sha256(local_path(root, case["image"]))}\n            record["cases"].append(item)\n            write_json(run_dir / "run.json", record)\n            start = time.perf_counter()\n            try:\n                garment = load_rgb(local_path(root, case["image"]))\n                torch.cuda.reset_peak_memory_stats()\n                torch.cuda.synchronize()\n                output = pipeline(person_image=person, garment_image=garment,\n                                  category=case["category"], garment_photo_type=case["garment_photo_type"], **record["settings"])\n                torch.cuda.synchronize()\n                if not output.images:\n                    raise RuntimeError("Model boş çıktı döndürdü.")\n                destination = run_dir / f\'{case["id"]}.png\'\n                output.images[0].save(destination)\n                item.update(status="completed", output=destination.name, output_sha256=sha256(destination),\n                            peak_allocated_gib=round(torch.cuda.max_memory_allocated() / 2**30, 3),\n                            peak_reserved_gib=round(torch.cuda.max_memory_reserved() / 2**30, 3))\n                # Presentation failure cannot erase the existence of a real inference output.\n                try:\n                    make_comparison(person, garment, output.images[0], run_dir / f\'{case["id"]}-comparison.jpg\')\n                except Exception as error:\n                    item["comparison_error"] = str(error)\n            except Exception as error:\n                item.update(status="failed", error_type=type(error).__name__, error=str(error),\n                            traceback=traceback.format_exc())\n                torch.cuda.empty_cache()\n                if isinstance(error, torch.cuda.OutOfMemoryError):\n                    record["stopped_reason"] = "out_of_memory"\n            finally:\n                item["elapsed_seconds"] = round(time.perf_counter() - start, 3)\n                write_json(run_dir / "run.json", record)\n            if record.get("stopped_reason"):\n                break\n        attempted = {c["id"] for c in record["cases"]}\n        record["cases"].extend({"id": c["id"], "status": "not_run", "quality_status": "not_reviewed"}\n                               for c in cases if c["id"] not in attempted)\n        record["status"] = "completed" if all(c["status"] == "completed" for c in record["cases"]) else "incomplete"\n    except Exception as error:\n        record.update(status="setup_failed", error_type=type(error).__name__, error=str(error), traceback=traceback.format_exc())\n        raise\n    finally:\n        write_json(run_dir / "run.json", record)\n        create_review(run_dir, record)\n        print(f"Deney kaydı: {run_dir}")\n    return run_dir\n\n\ndef create_review(run_dir, record):\n    """Blank scores always stay blank; generation is never a quality pass."""\n    path = Path(run_dir) / "review.csv"\n    if path.exists():\n        return\n    with path.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=["id", "reviewer", *METRICS, "untargeted_changes", "critical_failure", "notes"])\n        writer.writeheader()\n        writer.writerows({"id": c["id"]} for c in record["cases"])\n\n\ndef evaluate(run_dir):\n    run_dir = Path(run_dir)\n    record = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))\n    with (run_dir / "review.csv").open(encoding="utf-8-sig", newline="") as stream:\n        rows = list(csv.DictReader(stream))\n    reviews = {}\n    for row in rows:\n        if row["id"] in reviews:\n            raise ValueError("Aynı deney için birden fazla değerlendirme satırı var.")\n        reviews[row["id"]] = row\n    decisions = []\n    for case in record["cases"]:\n        row = reviews.get(case["id"], {})\n        decision = "not_reviewed"\n        if case["status"] != "completed":\n            decision = "not_generated"\n        elif row.get("critical_failure", "").strip().lower() == "yes" or row.get("untargeted_changes", "no").strip().lower() == "yes":\n            decision = "failed"\n        elif row.get("reviewer", "").strip() and row.get("critical_failure", "").strip().lower() == "no" and row.get("untargeted_changes", "no").strip().lower() == "no":\n            raw = [row.get(metric, "").strip() for metric in METRICS]\n            if all(value in {"0", "1", "2", "3", "4"} for value in raw):\n                decision = "passed" if min(map(int, raw)) >= 3 else "failed"\n        decisions.append({"id": case["id"], "decision": decision, "review": row})\n    manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))\n    expected = {c["id"] for c in manifest["cases"]}\n    complete_suite = len(expected) == 10 and {c["id"] for c in record["cases"]} == expected\n    passed = sum(d["decision"] == "passed" for d in decisions)\n    pending = any(d["decision"] == "not_reviewed" for d in decisions)\n    gate = "pending" if pending or not complete_suite else ("passed" if passed >= 8 else "failed")\n    summary = {"quality_gate": gate, "passed": passed, "expected": 10,\n               "complete_suite": complete_suite, "decisions": decisions,\n               "scope": "Tek hedef kişi; genel kullanıcı başarısı veya beden uyumu kanıtı değildir."}\n    write_json(run_dir / "quality.json", summary)\n    sections = ["<!doctype html><html lang=\'tr\'><meta charset=\'utf-8\'><title>TryFit Next deney raporu</title>",\n                "<style>body{font:16px system-ui;max-width:1100px;margin:32px auto;padding:16px}img{max-width:100%}pre{white-space:pre-wrap}</style>",\n                "<h1>TryFit Next — insan değerlendirmesi</h1>",\n                f"<p>Kalite eşiği: {gate}. Geçen: {passed}/10.</p>"]\n    for item in decisions:\n        identifier = item["id"]\n        sections.append(f"<h2>{html.escape(identifier)}: {item[\'decision\']}</h2>")\n        comparison = run_dir / f"{identifier}-comparison.jpg"\n        if comparison.exists():\n            sections.append(f\'<img src="{html.escape(comparison.name, quote=True)}" alt="Kaynak, hedef, sonuç">\')\n        sections.append("<pre>" + html.escape(json.dumps(item["review"], ensure_ascii=False, indent=2)) + "</pre>")\n    sections.append("</html>")\n    (run_dir / "report.html").write_text("\\n".join(sections), encoding="utf-8")\n    return summary\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser(description="TryFit Next araştırma araçları")\n    commands = parser.add_subparsers(dest="command", required=True)\n    review_parser = commands.add_parser("evaluate", help="Doldurulmuş review.csv dosyasını değerlendir")\n    review_parser.add_argument("run_dir", type=Path)\n    run_parser = commands.add_parser("run", help="CUDA ortamında gerçek inference")\n    run_parser.add_argument("--root", type=Path, required=True)\n    run_parser.add_argument("--manifest", type=Path, required=True)\n    run_parser.add_argument("--weights", type=Path, required=True)\n    run_parser.add_argument("--case", action="append", dest="cases")\n    args = parser.parse_args()\n    if args.command == "evaluate":\n        print(json.dumps(evaluate(args.run_dir), ensure_ascii=False, indent=2))\n    else:\n        run_experiment(args.root, args.manifest, args.weights, selected_ids=args.cases)\n', encoding='utf-8')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import importlib
import experiment
importlib.reload(experiment)
environment = experiment.preflight()
print(json.dumps(environment, indent=2))
if 'CUDAExecutionProvider' not in environment['onnx_providers']:
    raise RuntimeError('ONNX CUDA provider eksik. CUDA/cuDNN uyumunu çözmeden ilerlemeyin.')


In [ ]:
# Ana model ve DWPose indirmelerini çözümlenen HF revision kimliklerine sabitle.
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
api = HfApi()
weights = ROOT / 'ai-worker/models'
weights.mkdir(parents=True, exist_ok=True)
provenance = {'upstream_commit': UPSTREAM_COMMIT, 'repositories': {}}
for repo, filenames, destination in [
    ('fashn-ai/fashn-vton-1.5', ['model.safetensors'], weights),
    ('fashn-ai/DWPose', ['yolox_l.onnx', 'dw-ll_ucoco_384.onnx'], weights / 'dwpose'),
]:
    revision = api.model_info(repo).sha
    provenance['repositories'][repo] = revision
    for filename in filenames:
        hf_hub_download(repo, filename=filename, revision=revision, local_dir=destination)
# Parser paketi main cache'ini kullandığı için aynı snapshot önceden indirilir ve hash'lenir.
parser_snapshot = Path(snapshot_download('fashn-ai/fashn-human-parser', revision='main'))
provenance['repositories']['fashn-ai/fashn-human-parser'] = parser_snapshot.name
provenance['parser_files_sha256'] = {
    str(p.relative_to(parser_snapshot)): experiment.sha256(p)
    for p in sorted(parser_snapshot.rglob('*')) if p.is_file()
}
experiment.write_json(ROOT / 'model-provenance.json', provenance)
print(json.dumps(provenance, indent=2))


## Girdileri yükleyin
Manifestte boş URL bırakmayın. Kaynak fotoğraf URL’si ile ürün sayfası URL’si ayrı alanlardır.
`person.consent_confirmed` yalnızca fotoğrafı kullanma izniniz varsa `true` olmalı.
Manifestte görseller `data/person.jpg`, `data/oversize-hoodie.jpg` vb. yolları kullanır.
Dosyaları bu adlarla hazırlayın. Farklı ad kullanırsanız manifestte değiştirin.
Ürün görsellerini otomatik kazımıyoruz; seçtiğiniz gerçek mankenli fotoğrafları yükleyin.


In [ ]:
from google.colab import files
data_dir = ROOT / 'data'
data_dir.mkdir(exist_ok=True)
manifest_path = ROOT / 'manifest.json'
if not manifest_path.exists():
    manifest_path.write_text('{\n  "schema_version": 1,\n  "person": {"image": "data/person.jpg", "consent_confirmed": false},\n  "cases": [\n    {"id": "oversize-hoodie", "label": "Oversize hoodie", "category": "tops", "garment_photo_type": "model", "image": "data/oversize-hoodie.jpg", "product_url": "", "source_image_url": "", "source_notes": ""},\n    {"id": "tshirt", "label": "Tişört", "category": "tops", "garment_photo_type": "model", "image": "data/tshirt.jpg", "product_url": "", "source_image_url": "", "source_notes": ""},\n    {"id": "patterned-blouse", "label": "Desenli bluz", "category": "tops", "garment_photo_type": "model", "image": "data/patterned-blouse.jpg", "product_url": "", "source_image_url": "", "source_notes": ""},\n    {"id": "trench-coat", "label": "Trençkot", "category": "tops", "garment_photo_type": "model", "image": "data/trench-coat.jpg", "product_url": "", "source_image_url": "", "source_notes": ""},\n    {"id": "denim-jacket", "label": "Kot ceket", "category": "tops", "garment_photo_type": "model", "image": "data/denim-jacket.jpg", "product_url": "", "source_image_url": "", "source_notes": ""},\n    {"id": "long-dress", "label": "Uzun elbise", "category": "one-pieces", "garment_photo_type": "model", "image": "data/long-dress.jpg", "product_url": "", "source_image_url": "", "source_notes": ""},\n    {"id": "fitted-dress", "label": "Dar elbise", "category": "one-pieces", "garment_photo_type": "model", "image": "data/fitted-dress.jpg", "product_url": "", "source_image_url": "", "source_notes": ""},\n    {"id": "wide-trousers", "label": "Geniş pantolon", "category": "bottoms", "garment_photo_type": "model", "image": "data/wide-trousers.jpg", "product_url": "", "source_image_url": "", "source_notes": ""},\n    {"id": "skirt", "label": "Etek", "category": "bottoms", "garment_photo_type": "model", "image": "data/skirt.jpg", "product_url": "", "source_image_url": "", "source_notes": ""},\n    {"id": "hooded-sweatshirt", "label": "Kapüşonlu sweatshirt", "category": "tops", "garment_photo_type": "model", "image": "data/hooded-sweatshirt.jpg", "product_url": "", "source_image_url": "", "source_notes": ""}\n  ]\n}\n', encoding='utf-8')
uploaded = files.upload()
for name, content in uploaded.items():
    basename = Path(name).name
    destination = manifest_path if basename == 'manifest.json' else data_dir / basename
    destination.write_bytes(content)
manifest = json.loads(manifest_path.read_text(encoding='utf-8-sig'))
print(json.dumps(manifest, ensure_ascii=False, indent=2))
# Eksik girdi varsa anlaşılır hata verir, inference başlamaz.
experiment.validate_manifest(manifest, ROOT)


## Önce bir ürünle doğrulayın
Bu deneme kalite eşiğini geçiremez; on ürünlük matris ayrıca çalıştırılmalıdır.
Hücre hata verse bile en alttaki arşiv indirme hücresini çalıştırarak logları alabilirsiniz.


In [ ]:
smoke_run = experiment.run_experiment(
    ROOT, manifest_path, weights, selected_ids=['tshirt'], seed=42, steps=30)
smoke_record = json.loads((smoke_run / 'run.json').read_text())
if smoke_record['status'] != 'completed':
    raise RuntimeError('İlk üretim başarısız. Logları inceleyin; toplu denemeye geçmeyin.')
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(smoke_run / 'tshirt-comparison.jpg')))


## On ürünlük ana deney
İlk çıktıyı inceleyip ortamın çalıştığını gördükten sonra bu hücreyi çalıştırın.
OOM hatası deneyi durdurur; kalan ürünler `not_run` olur. Başarısız sonuç yerine başka görsel konmaz.


In [ ]:
if 'smoke_record' not in globals() or smoke_record['status'] != 'completed':
    raise RuntimeError('Önce tek ürün üretimini tamamlayın.')
batch_run = experiment.run_experiment(ROOT, manifest_path, weights, seed=42, steps=30)
experiment.evaluate(batch_run)
for comparison in sorted(batch_run.glob('*-comparison.jpg')):
    display(DisplayImage(filename=str(comparison)))
files.download(str(batch_run / 'review.csv'))


## İnsan değerlendirmesi
`review.csv` dosyasında her metriği 0–4 puanlayın: 0 yanlış/kullanılamaz, 1 ağır bozulma,
2 belirgin hata, 3 kabul edilebilir küçük hata, 4 güçlü koruma.
`source_pose`: 4 kaynak pozundan belirgin hata yok, 0 ağır kaynak poz/kaynak kişi aktarımı.
`reviewer` adını, `critical_failure` ve `untargeted_changes` alanlarına `yes` veya `no` ve hata notlarını girin.
Seçilmemiş kıyafetin değişmesi veya kaynak ayakkabı/aksesuarının taşınması `untargeted_changes=yes` olur ve örnek başarısız sayılır.
Yanlış kıyafet, belirgin yüz değişimi veya anatomi bozulması kritik hatadır.
Boş değerlendirme başarı değildir. Her metriği en az 3 olan ve kritik hatası olmayan örnek geçer.
10 örneğin en az 8'i geçerse önerilen araştırma eşiği geçilir; bu genel kullanıcı/beden garantisi değildir.


In [ ]:
review_upload = files.upload()
if set(review_upload) != {'review.csv'}:
    raise ValueError('Yalnızca doldurduğunuz review.csv dosyasını seçin.')
(batch_run / 'review.csv').write_bytes(review_upload['review.csv'])
print(json.dumps(experiment.evaluate(batch_run), ensure_ascii=False, indent=2))


In [ ]:
# Hata oluşmuş olsa da tüm deney kayıtlarını indirir. Fotoğrafları özel tutun.
import shutil
if not (ROOT / 'runs').exists():
    raise RuntimeError('Henüz deney kaydı yok.')
archive = shutil.make_archive('/content/tryfit-next-results', 'zip', ROOT, 'runs')
files.download(archive)
